# Spark Exercise

Apache Spark is an excellent tool for data engineering projects due to its robust ability to process large-scale data efficiently through distributed computing. Spark's in-memory processing capabilities significantly enhance the speed of data operations, making it ideal for handling big data workloads. It supports various data sources and formats, offering versatility in data ingestion and transformation. Additionally, Spark's rich API supports multiple programming languages such as Python, Java, and Scala, catering to diverse developer preferences. Its ecosystem, which includes libraries for SQL, machine learning, and graph processing, provides a comprehensive suite for building complex data pipelines and analytics, making it a powerful and flexible choice for data engineering tasks.

Use Python, ```pyspark``` and ```pandas``` to explore Apache Spark RDD and DataFrame:

# Spark RDD

Spark RDD (Resilient Distributed Dataset) is a fundamental data structure in Apache Spark that enables fault-tolerant, distributed processing of large datasets across multiple nodes in a cluster. Spark RDDs provide a higher-level abstraction for performing distributed data processing tasks, including both map (transformations) and reduce (aggregations) operations.

## Import Necessary Libraries

In [1]:
import csv
import os
import sys
from io import StringIO

import pandas as pd
from pyspark import SparkConf, SparkContext
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, count
from pyspark.sql.types import DoubleType, IntegerType, StringType, StructField, StructType

## Spark Context and Session
Initialize Spark Context and Spark Session

In [2]:
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)
os.environ.setdefault("PYSPARK_DRIVER_PYTHON", sys.executable)

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

csv_file = "cleaned-amazon-products-electronics-sales-2023.csv"

conf = SparkConf()
conf = conf.setMaster("local[2]")
conf = conf.setAppName("bdeng-spark")
conf = conf.set("spark.ui.showConsoleProgress", "false")

sc = SparkContext.getOrCreate(conf=conf)
sc.setLogLevel("ERROR")

spark_builder = SparkSession.builder
spark_builder = spark_builder.appName("spark")
spark = spark_builder.getOrCreate()

print(sc.version)
print(sc.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/08 01:05:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


4.1.2
local[2]


In [3]:
def parse_csv_line(line):
    line_as_file = StringIO(line)
    csv_reader = csv.reader(line_as_file)
    row = next(csv_reader)
    return row

def get_rating_pair(row):
    rating = row[5]
    return rating, 1

def add_numbers(first_number, second_number):
    return first_number + second_number

def sort_by_rating(item):
    rating = item[0]
    return rating

## Load Data into RDD

In [4]:
lines_rdd = sc.textFile(csv_file)
header = lines_rdd.first()

def is_not_header(line):
    return line != header

data_lines_rdd = lines_rdd.filter(is_not_header)
products_rdd = data_lines_rdd.map(parse_csv_line)

print(products_rdd.count())

9411


## Map Operation

Split data into individual parts and create key-value pairs

In [5]:
rating_pairs_rdd = products_rdd.map(get_rating_pair)

print(rating_pairs_rdd.take(5))

[('4.0', 1), ('4.3', 1), ('4.2', 1), ('4.1', 1), ('4.3', 1)]


## Reduce Operation

Reduce your key-value pairs

In [6]:
rating_counts_rdd = rating_pairs_rdd.reduceByKey(add_numbers)

## Collect Results

Because of lazy evaluation, the map-reduce operation is performed only now. Show what you calculated.

We calculated the number of products for each rating.

In [7]:
rating_counts = rating_counts_rdd.collect()
rating_counts = sorted(rating_counts, key=sort_by_rating)

for rating, product_count in rating_counts:
    print(rating, product_count)

1.0 8
1.3 1
1.4 1
1.5 2
1.7 2
1.8 1
1.9 1
2.0 2
2.1 3
2.2 4
2.3 3
2.4 6
2.5 7
2.6 9
2.7 7
2.8 18
2.9 25
3.0 44
3.1 45
3.2 61
3.3 102
3.4 162
3.5 203
3.6 309
3.7 397
3.8 610
3.9 793
4.0 973
4.1 1155
4.2 1209
4.3 1301
4.4 873
4.5 583
4.6 251
4.7 95
4.8 39
4.9 9
5.0 97


## Save Results

In [8]:
rating_counts_rdd.saveAsTextFile("saved-rating-rdd")
print("RDD saved.")

RDD saved.


# Spark DataFrame

Spark DataFrame is a distributed collection of data organized into named columns, designed for efficient data manipulation and analysis in Apache Spark. It is used for various data processing tasks such as data ingestion, transformation, querying, and analysis in Apache Spark, providing a high-level abstraction that simplifies working with structured data.

## Load Data into DataFrame

In [9]:
schema = StructType([
    StructField("name", StringType(), True),
    StructField("main_category", StringType(), True),
    StructField("sub_category", StringType(), True),
    StructField("image", StringType(), True),
    StructField("link", StringType(), True),
    StructField("ratings", DoubleType(), True),
    StructField("no_of_ratings", IntegerType(), True),
    StructField("discount_price", DoubleType(), True),
    StructField("actual_price", DoubleType(), True),
])

df = (
    spark.read
    .option("header", "true")
    .option("mode", "DROPMALFORMED")
    .schema(schema)
    .csv(csv_file)
)

## View DataFrame Schema

In [10]:
df.printSchema()

root
 |-- name: string (nullable = true)
 |-- main_category: string (nullable = true)
 |-- sub_category: string (nullable = true)
 |-- image: string (nullable = true)
 |-- link: string (nullable = true)
 |-- ratings: double (nullable = true)
 |-- no_of_ratings: integer (nullable = true)
 |-- discount_price: double (nullable = true)
 |-- actual_price: double (nullable = true)



## View DataFrame Data

In [11]:
df.show(10)

+--------------------+-------------------+---------------+--------------------+--------------------+-------+-------------+--------------+------------+
|                name|      main_category|   sub_category|               image|                link|ratings|no_of_ratings|discount_price|actual_price|
+--------------------+-------------------+---------------+--------------------+--------------------+-------+-------------+--------------+------------+
|Redmi 10 Power (P...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    4.0|          965|        120.99|      208.99|
|OnePlus Nord CE 2...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    4.3|       113956|        208.99|      219.99|
|OnePlus Bullets Z...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    4.2|        90304|         21.99|       25.29|
|Samsung Galaxy M3...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.ama

## Filter Data

Performe a filter operation on a column

We counted all the products with a discount price < 20 EUR.

In [12]:
cheap_products_df = df.filter(df.discount_price < 20.0)

print(cheap_products_df.count())
cheap_products_df.show(10)

7349
+--------------------+-------------------+---------------+--------------------+--------------------+-------+-------------+--------------+------------+
|                name|      main_category|   sub_category|               image|                link|ratings|no_of_ratings|discount_price|actual_price|
+--------------------+-------------------+---------------+--------------------+--------------------+-------+-------------+--------------+------------+
|boAt Airdopes 141...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    3.9|       172347|         14.29|       43.94|
|Apple 20W USB-C P...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    4.6|        61748|         17.48|        20.9|
|boAt Airdopes Ato...|tv, audio & cameras|All Electronics|https://m.media-a...|https://www.amazo...|    3.8|         3327|         14.29|       49.39|
|boAt Airdopes 141...|tv, audio & cameras|All Electronics|https://m.media-a...|https://ww

## Group By and Aggregate

Performe a group by and aggregat operation

In [13]:
grouped_by_rating_df = df.groupBy(df.ratings)

rating_summary_df = grouped_by_rating_df.agg(
    count("*").alias("product_count"),
    avg(df.discount_price).alias("avg_discount_price"),
)

rating_summary_df = rating_summary_df.sort("ratings")

rating_summary_df.show()

+-------+-------------+------------------+
|ratings|product_count|avg_discount_price|
+-------+-------------+------------------+
|    1.0|            8|           6.63125|
|    1.3|            1|              7.69|
|    1.4|            1|              5.82|
|    1.5|            2|             9.285|
|    1.7|            2|             13.69|
|    1.8|            1|              2.19|
|    1.9|            1|              3.58|
|    2.0|            2|             2.905|
|    2.1|            3|              5.53|
|    2.2|            4|             5.255|
|    2.3|            3| 9.486666666666666|
|    2.4|            6|14.295000000000002|
|    2.5|            7| 6.029999999999999|
|    2.6|            7| 84.44714285714285|
|    2.7|            7| 7.817142857142857|
|    2.8|           18|19.789999999999996|
|    2.9|           25|14.839199999999998|
|    3.0|           43|30.374186046511625|
|    3.1|           44| 26.37636363636364|
|    3.2|           58| 16.92827586206897|
+-------+--

## Save DataFrame to Parquet

In [14]:
df.write.parquet("saved-products-dataframe")
print("DataFrame saved.")

DataFrame saved.


In [15]:
spark.stop()